# What Goes With What? Market Basket Analysis with Association Rules in Python
### IST 4520 · Association Rules · CSUSB

**After this lecture you will be able to:**

1. Explain what a transaction, an item, an itemset and a rule are, and read a rule such as `(Chips) → (Salsa)`.
2. Turn a list of receipts into the True/False table that association-rule tools expect.
3. Run the two `mlxtend` functions — `apriori()` and `association_rules()` — and read their output tables.
4. **Interpret support, confidence and lift in plain English**, and tell a real pairing from a false alarm.
5. Explain why the Apriori algorithm is needed and what `min_support` controls.

**How to use this notebook:** read each text cell, then run the code cell under it (click the cell, press **Shift + Enter**). Run the cells in order — later cells use variables created by earlier ones. Sections marked **⏭️ Optional** (the math behind the numbers, Apriori written from scratch) are for reading at home. Everything runs in Google Colab (**Runtime ▸ Run all**).

# Section 1. What is an association rule?

An **association rule** is an *if–then* statement about things that occur together:

> **{Chips} → {Salsa}** — "receipts that contain chips tend to contain salsa too."

The left side is the **antecedent** (*if*), the right side the **consequent** (*then*).

The classic use is **market basket analysis**: a store takes thousands of receipts and asks *"what goes with what?"* — then uses the answers for shelf placement, bundles, coupons, and *"frequently bought together."* The same method works on anything where items occur together: songs in a playlist, courses in a schedule, symptoms in a patient record, alerts in a security log.

It is **unsupervised learning**: there is no answer column to predict. The algorithm finds the patterns; *we* judge which ones matter.

### Where this fits among the methods in this course

| Method | Input | What it finds | Is there a target column? |
|---|---|---|---|
| **Association rules** (today) | one row per receipt, one column per item (True/False) | *items* that go together | **No** — unsupervised |
| **k-means clustering** | one row per customer, numeric columns | *customers* that are similar to each other | **No** — unsupervised |
| **k-nearest neighbors** (classification) | the same customer table **plus a label** | a prediction of the label for a new customer | **Yes** — supervised |

> *The famous story:* a 1992 drugstore analysis found **beer and diapers** bought together on weekday evenings. The analysis was real (the part about moving the beer next to the diapers is mostly legend) — and it is a perfect picture of what this method does: it finds pairs nobody thought to look for.

# Section 2. The data: receipts → a True/False table

## 2.1 Setup: the Python packages we need

| Package | What we use it for |
|---|---|
| `pandas` | tables (DataFrames), sorting and filtering |
| `mlxtend` | `TransactionEncoder` (receipts → True/False table) · `apriori` (frequent itemsets + support) · `association_rules` (rules + confidence + lift) |

Colab has `pandas` built in. `mlxtend` ("machine learning extensions") is usually there too; the first cell installs it if it is missing.

In [1]:
try:
    import mlxtend                                   # try to load the package ...
except ImportError:                                  # ... and only if Python cannot find it:
    %pip install -q mlxtend                          #     install it (quietly) ...
    import mlxtend                                   #     ... and load it

print("mlxtend version:", mlxtend.__version__)       # which version we are running (useful when asking for help)
# If association_rules() later complains about a missing "num_itemsets" argument, you have mlxtend 0.23.2/0.23.3:
# run  %pip install -q -U mlxtend  and restart the runtime.

mlxtend version: 0.25.0


In [2]:
import pandas as pd                                                   # tables (DataFrames)

from mlxtend.preprocessing import TransactionEncoder                 # Step 0: receipts -> True/False table
from mlxtend.frequent_patterns import apriori, association_rules     # Step 1: frequent itemsets (support); Step 2: rules (confidence, lift)

# Colab's own Jupyter plumbing (jupyter_client) prints a harmless DeprecationWarning after every cell on Python 3.13.
# It is not caused by this notebook; this line hides it.
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

## 2.2 Fifteen receipts

Our toy data: 15 receipts from a campus convenience store. One receipt = one **transaction**; one product = one **item**; items that appear together form an **itemset** (`{Chips, Salsa}` is a *2-itemset*).

| Receipt | Items | Receipt | Items | Receipt | Items |
|---|---|---|---|---|---|
| T1 | Bread, Milk, Eggs | T6 | Bread, Milk, Eggs | T11 | Bread, Milk, Chips, Salsa |
| T2 | Bread, Milk, Eggs, Coffee | T7 | Milk, Eggs, Coffee | T12 | Bread, Milk, Eggs, Coffee |
| T3 | Milk, Chips, Salsa | T8 | Bread, Chips, Salsa | T13 | Bread, Chips |
| T4 | Bread, Milk, Coffee | T9 | Bread, Milk, Eggs | T14 | Bread, Milk, Eggs |
| T5 | Chips, Salsa | T10 | Milk, Eggs | T15 | Milk, Eggs |

In Python, a receipt is a list of items, and the data is a list of receipts:

In [3]:
receipts = [                                # a list of lists: one inner list per receipt (the order of items inside does not matter)
    ["Bread", "Milk", "Eggs"],              # T1
    ["Bread", "Milk", "Eggs", "Coffee"],    # T2
    ["Milk", "Chips", "Salsa"],             # T3
    ["Bread", "Milk", "Coffee"],            # T4
    ["Chips", "Salsa"],                     # T5
    ["Bread", "Milk", "Eggs"],              # T6
    ["Milk", "Eggs", "Coffee"],             # T7
    ["Bread", "Chips", "Salsa"],            # T8
    ["Bread", "Milk", "Eggs"],              # T9
    ["Milk", "Eggs"],                       # T10
    ["Bread", "Milk", "Chips", "Salsa"],    # T11
    ["Bread", "Milk", "Eggs", "Coffee"],    # T12
    ["Bread", "Chips"],                     # T13
    ["Bread", "Milk", "Eggs"],              # T14
    ["Milk", "Eggs"],                       # T15
]

len(receipts)                               # number of transactions (N = 15)

15

## 2.3 One-hot encoding: receipts → a True/False table

Algorithms want a table, not ragged lists: **one row per receipt, one column per item**, `True` if the item is on the receipt. This is called the *basket* (or *one-hot*) format.
`TransactionEncoder` does it in two steps — `fit` learns the list of items, `transform` converts the receipts.

In [4]:
te = TransactionEncoder()                                   # the converter object
te.fit(receipts)                                            # fit: learn the list of distinct items (it finds 6, sorted A-Z)
one_hot = te.transform(receipts)                            # transform: one True/False row per receipt
basket = pd.DataFrame(one_hot, columns=te.columns_)         # wrap it in a DataFrame; te.columns_ = the item names, in column order
basket.index = "T" + (basket.index + 1).astype(str)         # label the rows T1 ... T15 instead of 0 ... 14

basket                                                      # one row = one receipt, one column = one item

,Bread,Chips,Coffee,Eggs,Milk,Salsa
T1,True,False,False,True,True,False
T2,True,False,True,True,True,False
T3,False,True,False,False,True,True
T4,True,False,True,False,True,False
T5,False,True,False,False,False,True
T6,True,False,False,True,True,False
T7,False,False,True,True,True,False
T8,True,True,False,False,False,True
T9,True,False,False,True,True,False
T10,False,False,False,True,True,False


In [5]:
basket.sum()     # True counts as 1 -> number of receipts containing each item (Milk: 12 of 15)

Bread     10
Chips      5
Coffee     4
Eggs       9
Milk      12
Salsa      4
dtype: int64

In [6]:
basket.mean()    # mean of a True/False column = share of True = the SUPPORT of each single item (Milk: 12/15 = 0.80)

Bread     0.666667
Chips     0.333333
Coffee    0.266667
Eggs      0.600000
Milk      0.800000
Salsa     0.266667
dtype: float64

📌 **Exam point.** `.sum()` on a True/False column counts the receipts; `.mean()` gives the *fraction* of receipts — which is exactly the support of a single item.

# Section 3. Three numbers that make a rule interesting

Any two items can be written as a rule — {Eggs} → {Salsa} is a rule too, a useless one. Three numbers separate useful rules from noise.
We let `mlxtend` compute them; the formulas are in **⏭️ Appendix A**.

## 3.1 Support — *how often does it happen?*

**Support** is the share of all receipts that contain an itemset. support(Milk) = 0.80 means milk is on 80% of receipts (12 of 15); support({Bread, Milk}) = 0.53 means 53% of receipts (8 of 15) have both.

An itemset whose support is at least a cutoff we choose, **`min_support`**, is called a **frequent itemset**. `apriori()` finds all of them and reports their support:

In [7]:
# STEP 1 - apriori() COUNTS.
# It scans the receipts and records the support of every itemset that clears min_support.
# This is the only step that touches the data, and the expensive one (hence the Apriori pruning trick, Section 4).
# Support is what it produces. For a single item you could read support straight off the table
# (basket["Milk"].mean() = 0.80); apriori() is what you need once you want pairs and triples without counting by hand.
frequent_itemsets = apriori(basket, min_support=0.1, use_colnames=True, max_len=2)
#   basket            -> the True/False table from Section 2.3 (apriori needs this format)
#   min_support=0.1   -> keep itemsets found on at least 10% of receipts. As a COUNT: 0.1 x 15 = 1.5, so 2 or more receipts.
#                        min_support decides what can be found AT ALL: an itemset that fails it never reaches Step 2,
#                        so no rule can ever be made from it (try min_support=0.3 and (Chips, Salsa) disappears).   # good exam question
#   use_colnames=True -> show item names instead of column numbers
#   max_len=2         -> single items and pairs only, for now (so Step 2 can only make one-item -> one-item rules)

frequent_itemsets.sort_values("support", ascending=False)      # the result: one row per frequent itemset, biggest support first

,support,itemsets
4,0.800000,(Milk)
0,0.666667,(Bread)
3,0.600000,(Eggs)
15,0.600000,"(Eggs, Milk)"
9,0.533333,"(Milk, Bread)"
8,0.400000,"(Eggs, Bread)"
1,0.333333,(Chips)
2,0.266667,(Coffee)
5,0.266667,(Salsa)
14,0.266667,"(Coffee, Milk)"


Read it like this: `(Milk)` 0.80 → 12 of 15 receipts · `(Bread, Milk)` 0.53 → 8 of 15 · `(Chips, Salsa)` 0.27 → 4 of 15. Support is just a percentage.
(Itemsets are shown in parentheses; the items inside appear in no particular order.)

📌 **Exam point.** `min_support` is a *proportion*, not a count. To get the count, multiply by the number of receipts: 0.1 × 15 = 1.5 → "at least 2 receipts". On 10,000 receipts, 0.01 means "at least 100 receipts".

## 3.2 Confidence — *when X is bought, how often is Y bought too?*

**Confidence** of X → Y: among the receipts that contain X, the share that also contain Y. It is the "how reliable is the *if–then*?" number.

`association_rules()` turns the frequent itemsets into rules and computes confidence (and lift) for each:

In [8]:
# STEP 2 - association_rules() DIVIDES.
# It never looks at the receipts. Its input is the table from Step 1 (it needs the 'support' and 'itemsets' columns).
# For every frequent itemset it tries every split into antecedent -> consequent (X -> Y) and computes
#     confidence(X -> Y) = support(X and Y) / support(X)
#     lift(X -> Y)       = confidence(X -> Y) / support(Y)
# All three supports it needs are already in the Step 1 table: if (Chips, Salsa) is frequent, then (Chips) and
# (Salsa) are frequent too (the Apriori principle), so their rows exist. That is why the input must be apriori()'s output.
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
#   frequent_itemsets -> the Step 1 table
#   metric / min_threshold only FILTER: confidence and lift (and more) are computed for every rule regardless;
#   this call just keeps the rows with confidence >= 0.6.

cols = ["antecedents", "consequents", "support", "confidence", "lift"]      # the 5 columns we care about (the table has more - see below)
rules[cols].sort_values("confidence", ascending=False).round(3)             # show those 5 columns, most confident rule first, 3 decimals

,antecedents,consequents,support,confidence,lift
9,(Coffee),(Milk),0.267,1.000,1.250
10,(Eggs),(Milk),0.600,1.000,1.250
6,(Salsa),(Chips),0.267,1.000,3.000
4,(Bread),(Milk),0.533,0.800,1.000
7,(Chips),(Salsa),0.267,0.800,3.000
8,(Coffee),(Eggs),0.200,0.750,1.250
1,(Coffee),(Bread),0.200,0.750,1.125
11,(Milk),(Eggs),0.600,0.750,1.250
2,(Eggs),(Bread),0.400,0.667,1.000
5,(Milk),(Bread),0.533,0.667,1.000


- `(Bread) → (Milk)`: confidence **0.80** — 8 of the 10 bread receipts also have milk.
- `(Chips) → (Salsa)`: confidence **0.80** — 4 of the 5 chip receipts also have salsa.
- Confidence is **directional**: `(Milk) → (Bread)` is only 0.67 (8 of the 12 milk receipts have bread).

📌 **Exam point.** `(Bread) → (Milk)` and `(Milk) → (Bread)` are two different rules with the same support and different confidence.

**The other columns.** `association_rules()` returns more than our five columns:

In [9]:
list(rules.columns)      # every column association_rules() computed

['antecedents',
 'consequents',
 'antecedent support',
 'consequent support',
 'support',
 'confidence',
 'lift',
 'representativity',
 'leverage',
 'conviction',
 'zhangs_metric',
 'jaccard',
 'certainty',
 'kulczynski']

`antecedent support` and `consequent support` are the supports of X and of Y on their own (the numbers from `basket.mean()`). `leverage`, `conviction`, `zhangs_metric` and the rest are alternative strength measures — you can ignore them in this course. We always select `cols`.

### 🗳️ Quick poll

`(Bread) → (Milk)` and `(Chips) → (Salsa)` have **the same confidence, 0.80**. Are they equally interesting rules? Hands up for *yes*… for *no*…

## 3.3 Lift — *is it better than chance?*

Milk is on **80% of all receipts**. So *every* rule "→ (Milk)" looks confident, whether or not X has anything to do with milk.

**Lift** fixes this: it compares the confidence with how often Y is bought *in general*. Lift 3 means "Y is 3 times more likely when X is in the basket than for a typical shopper."

| Lift | Meaning |
|---|---|
| **≈ 1** | X and Y are **independent** — knowing X tells you nothing about Y |
| **> 1** | they appear together **more** than chance → positive association |
| **between 0 and 1** | they appear together **less** than chance → *negative association* (the items avoid each other) |
| **0** | never on the same receipt — the lowest possible value. **Lift is never a negative number.** |

Look at the `lift` column of the same rules:

In [10]:
rules[cols].sort_values("lift", ascending=False).round(3)     # same table as above, now sorted by lift - lift was computed in Step 2 along with confidence

,antecedents,consequents,support,confidence,lift
6,(Salsa),(Chips),0.267,1.000,3.000
7,(Chips),(Salsa),0.267,0.800,3.000
8,(Coffee),(Eggs),0.200,0.750,1.250
9,(Coffee),(Milk),0.267,1.000,1.250
10,(Eggs),(Milk),0.600,1.000,1.250
11,(Milk),(Eggs),0.600,0.750,1.250
1,(Coffee),(Bread),0.200,0.750,1.125
2,(Eggs),(Bread),0.400,0.667,1.000
3,(Bread),(Eggs),0.400,0.600,1.000
4,(Bread),(Milk),0.533,0.800,1.000


Same confidence, very different stories:

- `(Chips) → (Salsa)`: lift **3.0** — salsa is on 27% of all receipts, but on 80% of chip receipts. Chip buyers are 3× more likely than the average shopper to buy salsa. **A real pairing.**
- `(Bread) → (Milk)`: lift **1.0** — milk is on 80% of all receipts and on 80% of bread receipts. Bread buyers buy milk at exactly the overall rate. The 0.80 confidence was a **false alarm**: milk is just popular.

📌 **Exam point.** Two rules with the same confidence can have completely different lifts. Confidence alone cannot tell a real pairing from a popular consequent; lift can.

**Q: Can lift be below 1?** Yes — items that *avoid* each other. We need the rules with low confidence too, so set no threshold and filter:

In [11]:
# Same Step 2, with no filter: metric="lift", min_threshold=0 keeps every rule the frequent itemsets can produce.
# (Rules are still limited to itemsets that passed min_support in Step 1 - nothing new is counted here.)
rules_all = association_rules(frequent_itemsets, metric="lift", min_threshold=0)
print(len(rules_all), "rules in total")                                                   # 22: each of the 11 frequent pairs gives 2 rules, one per direction

rules_all[cols][rules_all["lift"] < 1].sort_values("lift").round(3)                       # boolean filter: keep only the rows with lift below 1, smallest first

22 rules in total


,antecedents,consequents,support,confidence,lift
11,(Chips),(Milk),0.133,0.400,0.500
10,(Milk),(Chips),0.133,0.167,0.500
21,(Milk),(Salsa),0.133,0.167,0.625
20,(Salsa),(Milk),0.133,0.500,0.625
8,(Salsa),(Bread),0.133,0.500,0.750
9,(Bread),(Salsa),0.133,0.200,0.750
0,(Bread),(Chips),0.200,0.300,0.900
1,(Chips),(Bread),0.200,0.600,0.900


`(Chips) → (Milk)`: lift **0.5** — 40% of chip buyers buy milk, half the usual 80%. Chip buyers are *half* as likely as the average shopper to buy milk. A snack run is not a grocery run.
(Remember: lift is never negative. "Negative association" means a lift between 0 and 1.)

*Why 22 rules?* Each frequent pair gives two rules, one per direction (11 pairs × 2 = 22). In general an itemset with *k* items yields 2ᵏ − 2 rules, which is why rule tables grow quickly once triples appear.

## 3.4 Putting the three numbers together — how to interpret a rule

This section is the heart of the lecture. The software produces the numbers; **interpreting them is your job.**

### Three questions, always in this order

For any rule X → Y:

1. **How often does this happen?** → **support**. Turn it into a count (support × N receipts). This tells you *how many customers the rule involves* and *how much evidence you have*.
2. **How reliable is the if → then?** → **confidence**. "When X is bought, Y is bought __% of the time."
3. **Is it a real connection, or is Y just popular?** → **lift**. Compare the confidence with Y's usual rate. 1 = nothing there.

| Number | The question it answers | In our data |
|---|---|---|
| **support** | **How often does this happen?** The share of *all* receipts that contain both X and Y. High support = the pattern involves many customers; low support = a pattern within a small group. | {Bread, Milk}: 0.53 → on 8 of 15 receipts. {Chips, Salsa}: 0.27 → on 4 of 15. |
| **confidence** | **How reliable is the if → then?** When X is on the receipt, how often is Y there too? | Chips → Salsa: 0.80 → 4 of the 5 chip receipts have salsa. |
| **lift** | **Is this a real connection, or is Y just popular?** How often Y is bought by X-buyers, divided by how often Y is bought by everyone. 1 = no connection; 3 = three times more likely. | Salsa is on 27% of all receipts but on 80% of chip receipts → 0.80 / 0.27 = 3.0. |

### The four rules, read in full

| Rule | support | confidence | lift | How to read it |
|---|---|---|---|---|
| (Bread) → (Milk) | 0.53 | 0.80 | 1.0 | **Happens a lot, means nothing.** More than half of all receipts (8 of 15) have both bread and milk, and 80% of bread buyers buy milk — but 80% of *all* shoppers buy milk. Knowing that someone bought bread tells you nothing new about milk. The two items are simply both popular. |
| (Chips) → (Salsa) | 0.27 | 0.80 | 3.0 | **A real pairing on a quarter of the receipts.** 4 of 15 receipts have both. When chips are bought, salsa follows 80% of the time — three times the salsa rate of a typical shopper (27%). Support says *how many customers* the pattern involves (about a quarter of them); lift says the pattern is *not a coincidence*. A store would act on this: shelve them together, bundle them, coupon one to sell the other. |
| (Salsa) → (Chips) | 0.27 | 1.00 | 3.0 | **The same pairing, read in the other direction.** Lift is identical (3.0) because lift describes the *pair* — chips and salsa go together, whichever way you read it. Confidence differs because the *direction* differs: all 4 salsa buyers also bought chips (1.00), but only 4 of the 5 chip buyers bought salsa (0.80) — receipt T13 has chips and no salsa. In practice: salsa in the cart makes chips a sure thing; chips in the cart make salsa likely. |
| (Chips) → (Milk) | 0.13 | 0.40 | 0.5 | **Items that avoid each other.** Only 2 of 15 receipts have both. 40% of chip buyers buy milk — half the usual 80%. Chips signal a different kind of shopping trip (a snack run), so this rule tells a store *not* to expect milk in that basket. |

📌 **Exam point.** Lift is symmetric — lift(X → Y) = lift(Y → X). Confidence is not.

### A second worked example: (Eggs) → (Milk) — support 0.60, confidence 1.00, lift 1.25

- **support 0.60** → 9 of the 15 receipts contain both eggs and milk. The pattern is *widespread*.
- **confidence 1.00** → every single egg buyer also bought milk. Perfectly *reliable*.
- **lift 1.25** → but milk is on 80% of all receipts anyway, so "100% of egg buyers" is only 1.25 times the usual rate. Only *mildly special*.

Reading: a very common, perfectly reliable, but only mildly special pairing. **A confidence of 1.00 did not make it a strong rule** — lift did not back it up. Compare with (Chips) → (Salsa): less common (0.27), slightly less reliable (0.80), but far more special (3.0). If you had to pick one rule to act on, it is the chips rule.

### So what does "support 0.27, confidence 0.80, lift 3.0" really tell me?

Three different things, and you need all three:

- *support 0.27* — the pattern shows up on about a quarter of all receipts (4 of 15). It is not rare, and it is not universal: it involves a specific group of customers.
- *confidence 0.80* — within that group, the *if → then* is reliable: 4 out of 5 chip buyers take salsa.
- *lift 3.0* — and that 80% is far above the 27% you would see for shoppers in general, so the pairing is real, not a side effect of salsa being popular.

### An interpretation template

Fill in the blanks for any rule, then add a verdict:

> "**__% of all receipts** (about __ receipts) contain both X and Y. **When X is bought, Y is bought __% of the time** — **__ times** the usual rate for Y."
> Verdict: *real pairing* / *false alarm* / *avoidance* — and, for a real pairing, how widespread it is.

- (Chips) → (Salsa): "**27% of all receipts** (4 of 15) contain both chips and salsa. **When chips are bought, salsa is bought 80% of the time — 3 times** the usual rate for salsa." Verdict: a real pairing within about a quarter of the receipts.
- (Bread) → (Milk): "**53% of all receipts** (8 of 15) contain both bread and milk. **When bread is bought, milk is bought 80% of the time — 1.0 times** the usual rate for milk." Verdict: a false alarm; both items are simply popular.

That is the sentence to give a store manager. It contains all three numbers, and the verdict follows from the last one.

### Support and lift measure different things

| | **lift ≈ 1** | **lift well above 1** |
|---|---|---|
| **high support** | popular items that happen to co-occur — (Bread) → (Milk). Nothing to act on. | a strong *and* widespread pairing — the best case, and rare in practice |
| **low support** | noise | a strong pairing within a small group of customers — (Chips) → (Salsa). Act on it for that group. |

One more thing support tells you: **how much evidence there is.** With 15 receipts, support 0.27 means 4 receipts — fine for a toy example, thin for a real decision. On real data you would want a rule to rest on hundreds of receipts before trusting it. That is why `min_support` is the first number you set.

### Common misreadings

| Misreading | Why it is wrong |
|---|---|
| "Confidence 1.0 means a strong rule." | Confidence ignores how popular Y is. (Eggs) → (Milk) has confidence 1.00 and lift only 1.25. Always check lift. |
| "Lift 3 means 3% more likely." | Lift is a *ratio*: 3 **times** as likely, not 3% more. |
| "Support 0.27 for (Chips, Salsa) means 27% of chip buyers bought salsa." | Support is a share of **all** receipts. The share *among chip buyers* is the confidence (0.80). |
| "Low support means the rule is unimportant." | Low support means it involves fewer customers. It can still be a strong pairing for that group (lift 3.0). |
| "Lift 0.5 is a negative lift." | Lift is never negative. 0.5 means Y is *half* as likely as usual. "Negative association" = lift between 0 and 1. |

### What to do with each kind of rule

| Kind of rule | The numbers | Typical action |
|---|---|---|
| Real, widespread pairing | high support, lift clearly above 1 | shelf placement, bundles, "frequently bought together" |
| Real pairing, small group | low support, lift clearly above 1 | targeted promotion or recommendation for that group; check the count behind it first |
| False alarm | any support, high confidence, lift ≈ 1 | nothing — Y sells anyway; do not discount Y to "boost" X |
| Avoidance | lift below 1 | do not bundle; expect different shopping trips; the items may be substitutes |

Look at **all three** numbers, in order. A good rule has enough support to matter, enough confidence to act on, and a lift clearly above 1.

# Section 4. Why we need an algorithm: Apriori

We just called `apriori()`. Why does it need a clever algorithm at all — why not check every possible itemset?

**Q: How many itemsets are possible with our 6 items? With the 169 product categories of a real grocery store?**

In [12]:
n_items = basket.shape[1]                                                   # shape = (rows, columns); columns = items, so this is 6
print("items:", n_items, " -> possible itemsets:", 2**n_items - 1)          # every non-empty combination of n items: 2**n - 1
print("items: 169 -> possible itemsets:", f"{2**169 - 1:.1e}")            # a real grocery store, in scientific notation

items: 6  -> possible itemsets: 63
items: 169 -> possible itemsets: 7.5e+50


$7.5 \times 10^{50}$ is more than the number of atoms in planet Earth. Checking every itemset is impossible — and that is a *small* store.

**The Apriori principle** (Agrawal & Srikant, 1994):

> If an itemset is rare, every bigger itemset that contains it is rare too.

If {Salsa} is on only 27% of receipts, then {Salsa, Milk}, {Salsa, Milk, Eggs}, … can never be on more than 27%. So Apriori counts single items, keeps the frequent ones, builds pairs *only from those*, keeps the frequent pairs, builds triples only from those, and so on. Almost everything is pruned before it is ever counted. That is the whole trick behind `apriori()`.

Then `association_rules()` takes the surviving itemsets and writes out the rules — two steps, two functions. (*⏭️ Appendix B* shows both steps written from scratch in a few lines of Python.)

📌 **Exam point.** `min_support` controls Step 1 and therefore everything: a rule whose itemset fails `min_support` is never evaluated, however high its confidence or lift would have been. `min_threshold` only filters the rules that Step 1 made possible.

---

## The whole workflow in five lines

```python
receipts = [["Bread", "Milk"], ["Chips", "Salsa"], ...]                                    # 1. one list per transaction
basket = pd.DataFrame(te.fit(receipts).transform(receipts), columns=te.columns_)          # 2. True/False table (TransactionEncoder)
frequent_itemsets = apriori(basket, min_support=0.1, use_colnames=True)                  # 3. Step 1: frequent itemsets + support
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)     # 4. Step 2: rules + confidence + lift
rules[cols].sort_values("lift", ascending=False)                                         # 5. sort by lift, filter, INTERPRET
```

## Key takeaways

- An association rule is **{if} → {then}**, found from data with no answer column: unsupervised learning.
- **support** = how often the combination happens · **confidence** = how reliable the if → then is · **lift** = whether it is a real connection (1 = nothing there). Read all three, in that order, and turn support into a count.
- Two function calls do the work: `apriori()` finds the frequent itemsets, `association_rules()` turns them into rules with the three numbers. `min_support` decides what can be found at all.
- The software ranks; **you interpret** — real pairing, false alarm, or avoidance — and say it in one sentence a manager understands.
- The same two calls scale to real data: a grocery store's 9,835 receipts and 169 products run in under a second. The only new work is getting the data into the True/False table; everything after that is this notebook.

# ✅ Practice problems — reading the three numbers

Everything here uses only what we did above: the tables printed in Section 3, `sort_values`, a boolean filter like `rules_all["lift"] < 1`, and the two `mlxtend` calls. **Answers are shown right below each question — try first.**

## Part A — interpretation (no code needed)

**Q1) Which number answers which question?** Match each question to *support*, *confidence* or *lift*.
(a) "Out of all receipts, how many have both X and Y?" (b) "When X is bought, how often is Y bought too?" (c) "Is Y more likely with X than it is in general?"

*Hint: Section 3.4, "Three questions, always in this order."*

**Answer**
(a) support · (b) confidence · (c) lift.

---

**Q2) Fill in the interpretation template** for `(Salsa) → (Chips)` (support 0.27, confidence 1.00, lift 3.0) and for `(Milk) → (Bread)` (support 0.53, confidence 0.67, lift 1.0). Then give each a verdict.

*Hint: the template in Section 3.4; the numbers are in the `rules` table of Section 3.2.*

**Answer**
- (Salsa) → (Chips): "27% of all receipts (4 of 15) contain both salsa and chips. When salsa is bought, chips are bought 100% of the time — 3 times the usual rate for chips (33%)." Verdict: a real pairing within about a quarter of the receipts; salsa in the cart makes chips a sure thing.
- (Milk) → (Bread): "53% of all receipts (8 of 15) contain both milk and bread. When milk is bought, bread is bought 67% of the time — 1.0 times the usual rate for bread (67%)." Verdict: a false alarm; milk tells you nothing about bread.

---

**Q3) One number at a time.** `(Coffee) → (Milk)` has support 0.267, confidence 1.00, lift 1.25. Write one sentence for each number, then say whether this is a strong pairing.

*Hint: compare the confidence with the base rate of milk (`basket.mean()`), exactly as in the (Eggs) → (Milk) example.*

**Answer**
- support 0.267 → 4 of the 15 receipts contain both coffee and milk: a pattern within a smallish group.
- confidence 1.00 → every coffee buyer (all 4) also bought milk: perfectly reliable.
- lift 1.25 → but 80% of *all* shoppers buy milk, so 100% is only 1.25 times the usual rate.
Verdict: reliable but only mildly special — milk is popular with everyone. Not a rule a store would build a promotion on.

---

**Q4) Same confidence, same lift, different support.** `(Eggs) → (Milk)` is 0.60 / 1.00 / 1.25 and `(Coffee) → (Milk)` is 0.27 / 1.00 / 1.25. What is the difference in meaning between the two rules? Which involves more customers?

*Hint: support is the only number that differs — what does support measure?*

**Answer**
Both rules are equally reliable (1.00) and equally (un)special (lift 1.25: milk is popular anyway). The difference is reach: the eggs rule appears on 60% of receipts (9 of 15), the coffee rule on 27% (4 of 15). The eggs pattern involves more than twice as many customers — but neither rule is a real connection beyond milk's popularity, so neither deserves action.

---

**Q5) Two rules from a different store** (made-up numbers):
Rule A: support 0.40, confidence 0.90, lift 1.0. Rule B: support 0.05, confidence 0.60, lift 4.0.
Describe each in words. Which one is the false alarm? Which one would you investigate further, and what would you check first?

*Hint: the 2×2 table "Support and lift measure different things" and the note on evidence.*

**Answer**
- Rule A: happens on 40% of receipts and the if → then holds 90% of the time — but lift 1.0 says Y is bought at exactly its usual rate whether or not X is in the basket. **False alarm**: Y is simply a popular item.
- Rule B: happens on only 5% of receipts, holds 60% of the time, and Y is **4 times** more likely with X than in general. A strong pairing within a small group of customers — worth investigating.
- Check first: the **count** behind the 5% (support × number of receipts). 5% of 200 receipts is 10 receipts (too thin to act on); 5% of 50,000 is 2,500 (plenty). Then decide what action fits that group.

---

**Q6) A rule below 1.** `(Chips) → (Milk)` has lift 0.5. Write what you would tell the store manager, and name one thing the store should *not* do because of it.

*Hint: Section 3.4, "Items that avoid each other" and the "What to do with each kind of rule" table.*

**Answer**
"Only 40% of chip buyers also buy milk, half the usual 80%. Chips go with a snack trip, not a grocery trip." The store should not bundle or cross-promote chips with milk, and should not expect milk in a chips basket.

---

**Q7) True or false?** Explain each answer in one sentence.
(a) "A confidence of 1.0 always means a strong rule."
(b) "Lift 3 means Y is 3% more likely with X."
(c) "Support 0.27 for (Chips, Salsa) means 27% of chip buyers bought salsa."
(d) "A negative association means the lift is below zero."

*Hint: the "Common misreadings" table.*

**Answer**
(a) False — (Eggs) → (Milk) has confidence 1.00 and lift only 1.25; confidence ignores how popular Y is.
(b) False — lift is a ratio: 3 *times* as likely (300%), not 3% more.
(c) False — support is a share of *all* receipts; the share among chip buyers is the confidence, 0.80.
(d) False — lift is never negative; "negative association" means lift between 0 and 1.

## Part B — with code (only idioms from the lecture)

**Q8)** Show all 22 rules in `rules_all` sorted by lift, then keep only the rules with lift ≥ 2. Which rules remain, and what do they have in common?

*Hint: `rules_all[cols].sort_values("lift", ascending=False)`, then a boolean filter like the one in Section 3.3 with `>= 2` instead of `< 1`.*

**Answer**
```python
rules_all[cols].sort_values("lift", ascending=False).round(3)        # all 22 rules, strongest first
rules_all[cols][rules_all["lift"] >= 2].round(3)                     # only (Chips) -> (Salsa) and (Salsa) -> (Chips), both lift 3.0
# Only the chips-salsa pair has a lift clearly above 1. Every other rule is between 0.5 and 1.25: milk, bread and eggs
# are popular items that pair with everything at roughly their usual rates.
```

---

**Q9)** Compare `(Milk) → (Chips)` with `(Chips) → (Milk)` in the lift-below-1 table from Section 3.3. Which numbers are the same, which differ, and why?

*Hint: which of the three numbers describe the pair, and which describes a direction?*

**Answer**
Same support (0.133 — the pair is on 2 receipts whichever way you read it) and same lift (0.5 — lift describes the pair). Different confidence: (Chips) → (Milk) is 0.40 (2 of the 5 chip receipts have milk) while (Milk) → (Chips) is 0.167 (2 of the 12 milk receipts have chips). Confidence depends on which item is the *if*.

---

**Q10)** Re-run `apriori()` on `basket` with `min_support=0.3` (no `max_len`). Which itemsets survive? What happened to `(Chips, Salsa)` — and why can `association_rules()` never produce `(Chips) → (Salsa)` from this new table, even though its lift is 3.0?

*Hint: the comment block above `apriori()` in Section 3.1 — Step 1 decides what reaches Step 2.*

**Answer**
```python
apriori(basket, min_support=0.3, use_colnames=True)
# 8 itemsets: (Bread) 0.667, (Chips) 0.333, (Eggs) 0.6, (Milk) 0.8,
#             (Bread, Eggs) 0.4, (Bread, Milk) 0.533, (Milk, Eggs) 0.6, (Bread, Milk, Eggs) 0.4
# (Chips, Salsa) has support 0.267 < 0.3, so Step 1 drops it. association_rules() only splits itemsets that are in its
# input table - it never counts anything itself - so (Chips) -> (Salsa) cannot appear, however strong it is.
# Salsa and Coffee (0.267 each) disappear entirely. min_support decides what can be found at all.
```

# ⏭️ Appendix A — the math behind the three numbers

$N$ = number of receipts. For an itemset $X$ and a rule $X \rightarrow Y$:

$$\text{support}(X) = \frac{\#\text{ receipts containing } X}{N}
\qquad\qquad
\text{support}(X \rightarrow Y) = \frac{\#\text{ receipts containing both } X \text{ and } Y}{N}$$

$$\text{confidence}(X \rightarrow Y) = \frac{\text{support}(X \rightarrow Y)}{\text{support}(X)} = P(Y \mid X)$$

$$\text{lift}(X \rightarrow Y) = \frac{\text{confidence}(X \rightarrow Y)}{\text{support}(Y)}
= \frac{\text{support}(X \rightarrow Y)}{\text{support}(X)\cdot\text{support}(Y)}$$

Lift is symmetric — lift(X → Y) = lift(Y → X) — because the last form does not change when X and Y swap. Confidence is not.

**By hand, for (Bread) → (Milk)** on the 15 receipts: support(Bread) = 10/15 = 0.667, support(Milk) = 12/15 = 0.8, support(Bread, Milk) = 8/15 = 0.533.
confidence = 0.533 / 0.667 = **0.80**; lift = 0.80 / 0.80 = **1.0**.

**For (Chips) → (Salsa):** support(Chips) = 5/15, support(Salsa) = 4/15, support(Chips, Salsa) = 4/15.
confidence = 4/5 = **0.80**; lift = 0.80 / (4/15) = **3.0**.

The same arithmetic in pandas — `&` is "both True", and the mean of a True/False column is the share of True:

In [13]:
def rule_metrics(basket, X, Y):
    """Support, confidence and lift of the rule X -> Y, computed from the True/False table. X and Y are lists of items."""
    has_X  = basket[X].all(axis=1)          # True if the receipt contains every item in X   (.all over the columns in X)
    has_Y  = basket[Y].all(axis=1)          # True if the receipt contains every item in Y
    has_XY = has_X & has_Y                  # True if it contains all of X and all of Y      (& = both True)

    support    = has_XY.mean()              # share of receipts with X and Y
    confidence = has_XY.sum() / has_X.sum() # receipts with X and Y, divided by receipts with X
    lift       = confidence / has_Y.mean()  # confidence divided by the share of receipts with Y (the base rate of Y)
    return {"rule": f"{X} -> {Y}", "support": support, "confidence": confidence, "lift": lift}


pd.DataFrame([                              # one row per rule
    rule_metrics(basket, ["Bread"], ["Milk"]),
    rule_metrics(basket, ["Chips"], ["Salsa"]),
    rule_metrics(basket, ["Salsa"], ["Chips"]),
    rule_metrics(basket, ["Chips"], ["Milk"]),
]).round(3)

,rule,support,confidence,lift
0,['Bread'] -> ['Milk'],0.533,0.8,1.0
1,['Chips'] -> ['Salsa'],0.267,0.8,3.0
2,['Salsa'] -> ['Chips'],0.267,1.0,3.0
3,['Chips'] -> ['Milk'],0.133,0.4,0.5


Exactly the numbers `association_rules()` gave us in Section 3.

# ⏭️ Appendix B — Apriori from scratch

**Brute force:** check every one of the 63 possible itemsets of our 6 items.

In [14]:
from itertools import combinations                   # combinations(items, k): every k-item subset of items

min_support = 0.1
items = list(basket.columns)                         # the 6 item names
frequent_bruteforce = []                             # we will collect the frequent itemsets here
checked = 0                                          # how many itemsets we counted

for k in range(1, len(items) + 1):                   # itemset sizes 1, 2, ..., 6
    for itemset in combinations(items, k):           # every itemset of that size
        checked += 1
        if basket[list(itemset)].all(axis=1).mean() >= min_support:    # support of this itemset (share of receipts with all its items)
            frequent_bruteforce.append(itemset)

print("Brute force checked", checked, "itemsets and found", len(frequent_bruteforce), "frequent ones.")

Brute force checked 63 itemsets and found 24 frequent ones.


**Apriori:** level by level, counting only candidates whose smaller subsets are all frequent.

In [15]:
def apriori_by_hand(basket, min_support):
    frequent = {}                                              # itemset -> support, for every frequent itemset found
    level = [frozenset([item]) for item in basket.columns]     # size-1 candidates: every single item
    k = 1
    while level:                                               # stop when a level has no candidates left
        supports = {c: basket[list(c)].all(axis=1).mean() for c in level}         # count each candidate
        keep = {c: s for c, s in supports.items() if s >= min_support}            # keep the frequent ones
        print(f"size {k}: {len(level):>2} candidates counted -> {len(keep):>2} frequent")
        frequent.update(keep)

        # next level: merge frequent itemsets into (k+1)-itemsets, but only if EVERY k-subset is frequent (the Apriori principle)
        candidates = set()
        for a in keep:
            for b in keep:
                union = a | b                                                                   # merge two frequent itemsets
                if len(union) == k + 1 and all(frozenset(sub) in keep for sub in combinations(union, k)):
                    candidates.add(union)                                                       # keep as a candidate only if all its k-subsets are frequent
        level = list(candidates)
        k += 1
    return frequent


frequent_by_hand = apriori_by_hand(basket, min_support=0.1)
print(len(frequent_by_hand), "frequent itemsets in total")

size 1:  6 candidates counted ->  6 frequent
size 2: 15 candidates counted -> 11 frequent
size 3:  8 candidates counted ->  6 frequent
size 4:  1 candidates counted ->  1 frequent
24 frequent itemsets in total


Same 24 frequent itemsets, but Apriori counted **6 + 15 + 8 + 1 = 30** candidates instead of 63, and it never looked at a 5- or 6-itemset. On 169 items the savings are astronomical.

`mlxtend`'s `apriori()` without `max_len` finds the same 24:

In [16]:
apriori(basket, min_support=0.1, use_colnames=True).shape      # (rows, columns) = (24 frequent itemsets, 2 columns)

(24, 2)

# Sources

*Algorithm: Agrawal & Srikant, "Fast Algorithms for Mining Association Rules," VLDB 1994. Library: [mlxtend](https://rasbt.github.io/mlxtend/) by Sebastian Raschka.*